# Diagnosis Report — Banco_de_sangre,_Hospital_General_de_Medellín_20260821


**Student**: Alba Luz Preciado Pinillo ; Alina Maria Alvarez
**Dataset**: Banco_de_sangre,_Hospital_General_de_Medellín_20260821
**Date**: 28/08/2026


## 0. Setup





In [5]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, re, unicodedata
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')
print('pandas', pd.__version__)

pandas 3.0.5


## 1. Carga

El archivo llega con **todas las columnas como texto**. Se cargan como `str` para no
perder información en el parseo automático, y se convierten explícitamente después.


In [6]:
RUTA = 'Banco_de_sangre,_Hospital_General_de_Medellín_20260821.csv' 

df = pd.read_csv(RUTA, dtype=str, encoding='utf-8')
print(f'CHECK 1 — Forma: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Memoria: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
df.head()

CHECK 1 — Forma: 35,840 filas x 11 columnas
Memoria: 20.49 MB


,ANO,TRIMESTRE,FECHA EXTRACCION,RH,BARRIO,CIUDAD,EDAD,ESTATURA,FECHA NACIMIENTO,PESO,SEXO
0,2020,4,28/12/2020,0+,POPULAR 1,MEDELLIN,41,NaN,1982 Jan 01 12:00:00 AM,NaN,M
1,2020,1,01/02/2020,0+,20 DE JULIO,MEDELLIN,43,1.74,1979 Mar 12 12:00:00 AM,80,M
2,2020,1,01/02/2020,0+,20 DE JULIO,MEDELLIN,43,1.74,1979 Mar 12 12:00:00 AM,80,M
3,2020,1,01/02/2020,0+,20 DE JULIO,MEDELLIN,44,1.6,1979 Apr 21 12:00:00 AM,86,F
4,2020,1,01/02/2020,0+,20 DE JULIO,MEDELLIN,44,1.6,1979 Apr 21 12:00:00 AM,86,F


In [7]:
from pathlib import Path
for f in Path('.').iterdir():
    print(repr(f.name))

'.git'
'.venv'
'Banco_de_sangre,_Hospital_General_de_Medellín_20260821.csv'
'diagnostico_datos.ipynb'
'README.md'
'requirements.txt'


## 2. Tipos: todo llega como texto

**CHECK 2** — las 11 columnas vienen como `object`. Cuatro son numéricas y dos son
fechas, en dos formatos distintos:

In [8]:
print('CHECK 2 — dtypes originales')
print(df.dtypes.value_counts().to_string())
print()

df['EDAD_n']     = pd.to_numeric(df['EDAD'], errors='coerce')
df['PESO_n']     = pd.to_numeric(df['PESO'], errors='coerce')
df['ESTATURA_n'] = pd.to_numeric(df['ESTATURA'], errors='coerce')
df['ANO_n']      = pd.to_numeric(df['ANO'], errors='coerce')
df['TRIM_n']     = pd.to_numeric(df['TRIMESTRE'], errors='coerce')
df['F_EXT'] = pd.to_datetime(df['FECHA EXTRACCION'], format='%d/%m/%Y', errors='coerce')
df['F_NAC'] = pd.to_datetime(df['FECHA NACIMIENTO'], format='%Y %b %d %I:%M:%S %p', errors='coerce')

for orig, nuevo in [('EDAD','EDAD_n'), ('PESO','PESO_n'), ('ESTATURA','ESTATURA_n'),
                    ('FECHA EXTRACCION','F_EXT'), ('FECHA NACIMIENTO','F_NAC')]:
    fallo = (df[nuevo].isna() & df[orig].notna()).sum()
    print(f'  {orig:<20} no convertidos: {fallo:,}')

CHECK 2 — dtypes originales
str    11

  EDAD                 no convertidos: 0
  PESO                 no convertidos: 0
  ESTATURA             no convertidos: 0
  FECHA EXTRACCION     no convertidos: 0
  FECHA NACIMIENTO     no convertidos: 0


**INFO columnas

In [9]:
pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'no_nulos': df.notna().sum(),
    'nulos': df.isna().sum(),
    'unicos': df.nunique(),
})

,dtype,no_nulos,nulos,unicos
ANO,str,35840,0,6
TRIMESTRE,str,35840,0,4
FECHA EXTRACCION,str,35840,0,1335
RH,str,35799,41,8
BARRIO,str,28052,7788,1274
CIUDAD,str,35840,0,28
EDAD,str,35838,2,55
ESTATURA,str,35761,79,72
FECHA NACIMIENTO,str,35838,2,13393
PESO,str,35799,41,99
